# Advanced Intrusion Detection System for SDN-IoT
## Traditional ML + Deep Learning + Custom + Real Federated Learning + Multi-Agent RL

**Dataset:** `ASEADOS_SDN_IoT.csv`

**Latest One (FL+RL):**
- Sections 1–8: same training pipeline as `IDS_Advanced_GPU.ipynb`
- Section 9: real Flower FedAvg — 1 server + 3 clients, **ResNet-MLP** (best federatable DL)
- Section 10: 5 RL agents from `COMPLETE_RL_FULL_CODE.ipynb`
- Sections 11–12: `Saved model/`; UI uses **only XGBoost** + live FL/RL proof panel



---
## 1. Setup, Installation & GPU Check

In [ ]:
# Run once. Installs everything the notebook needs.
# IMPORTANT for GPU: the default `torch` from PyPI is CPU-only. Install the CUDA build:
#   pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128 --force-reinstall --no-deps
# (cu128 supports the GTX 1650 / Turing sm_75. Adjust the cu1XX tag to match your driver.)

%pip install -q numpy pandas matplotlib seaborn plotly scikit-learn imbalanced-learn xgboost lightgbm catboost joblib flask flwr "gymnasium>=0.29" python-docx
# %pip install -q torch==2.10.0 --index-url https://download.pytorch.org/whl/cu128 --force-reinstall --no-deps

In [ ]:
import warnings, os, json, time, copy, math, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import joblib

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    SMOTE = None
    print('imbalanced-learn not installed — run the pip cell above')

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('xgboost not installed — run the pip cell above')

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('lightgbm not installed — run the pip cell above')

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except ImportError:
    HAS_CAT = False
    print('catboost not installed — run the pip cell above')

GPU_OK = torch.cuda.is_available()
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 120)

RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch        :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))
else:
    print('GPU            : NOT FOUND -> running on CPU. Install the CUDA torch build (see install cell).')
print('Device selected:', DEVICE)

In [ ]:
# ---- Project paths: models under "Saved model" ----
BASE_DIR = os.getcwd()
SAVE_DIR = os.path.join(BASE_DIR, 'Saved model')
TM_DIR   = os.path.join(SAVE_DIR, 'TM')
DL_DIR   = os.path.join(SAVE_DIR, 'DL')
RL_DIR   = os.path.join(SAVE_DIR, 'RL')
FL_SAVE  = os.path.join(SAVE_DIR, 'FL')
FL_DIR   = os.path.join(BASE_DIR, 'federated')
ART_DIR  = SAVE_DIR
for d in (TM_DIR, DL_DIR, RL_DIR, FL_SAVE, FL_DIR, ART_DIR):
    os.makedirs(d, exist_ok=True)

_candidates = [
    'ASEADOS_SDN_IoT.csv',
    os.path.join('..', 'ASEADOS_SDN_IoT.csv'),
    r'E:\CSE499B\ASEADOS_SDN_IoT.csv',
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), _candidates[0])
print('Dataset        ->', os.path.abspath(DATA_PATH))
print('Saved model    ->', SAVE_DIR)
print('Federated      ->', FL_DIR)



---
## 2. Data Loading & Inspection

In [ ]:
t0 = time.time()
df = pd.read_csv(DATA_PATH)
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
print('=== Label distribution ===')
display(df['Label'].astype(str).str.strip().value_counts().to_frame('count'))
print('\n=== Dtypes ===')
print(df.dtypes.value_counts())
na = df.isnull().sum()
print('\nColumns with NaN:', int((na > 0).sum()))

---
## 3. Data Cleaning & Preprocessing

Steps: drop identifier columns -> fix inf/NaN -> remove constant + highly-correlated features ->
strip & encode labels -> stratified 80/20 split -> Min-Max scaling -> SMOTE on the training set only.

In [ ]:
# Step 1 - drop identifier / leakage columns
drop_cols = [c for c in ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp'] if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print('Dropped identifiers:', drop_cols, '-> shape', df.shape)

In [ ]:
# Step 2 - infinities and missing values
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
n_na = int(df.isnull().sum().sum())
for c in num_cols:
    if df[c].isnull().any():
        df[c] = df[c].fillna(df[c].median())
print(f'Replaced inf and imputed {n_na:,} missing values with column medians.')

In [ ]:
# Step 3 - remove constant / near-zero-variance features
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
low_var = [c for c in num_cols if df[c].std() < 1e-10]
df.drop(columns=low_var, inplace=True)
print('Removed', len(low_var), 'constant features:', low_var)
print('Shape:', df.shape)

In [ ]:
# Step 4 - remove duplicate rows
dups = int(df.duplicated().sum())
if dups:
    df.drop_duplicates(inplace=True)
print(f'Removed {dups:,} duplicate rows -> shape {df.shape}')

In [ ]:
# Step 5 - clean + encode labels
df['Label'] = df['Label'].astype(str).str.strip()
label_encoder = LabelEncoder()
df['y'] = label_encoder.fit_transform(df['Label'])
CLASS_NAMES = label_encoder.classes_.tolist()
NUM_CLASSES = len(CLASS_NAMES)
print('Classes:', dict(zip(CLASS_NAMES, range(NUM_CLASSES))))

In [ ]:
# Step 6 - feature matrix (numeric only) + correlation pruning
exclude = ['Label', 'y']
feature_cols = [c for c in df.columns if c not in exclude]
feature_cols = [c for c in feature_cols if df[c].dtype != object]

corr = df[feature_cols].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.95)]
feature_cols = [c for c in feature_cols if c not in to_drop]
print(f'Dropped {len(to_drop)} highly-correlated (>0.95) features.')
print(f'Final feature count: {len(feature_cols)}')

X = df[feature_cols].values.astype(np.float32)
y = df['y'].values.astype(np.int64)

In [ ]:
# Step 7 - stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print('Train:', X_train.shape, ' Test:', X_test.shape)

In [ ]:
# Step 8 - Min-Max scaling (fit on train only)
scaler = MinMaxScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_test_s  = scaler.transform(X_test).astype(np.float32)
X_train_s = np.nan_to_num(X_train_s, nan=0.0, posinf=1.0, neginf=0.0)
X_test_s  = np.nan_to_num(X_test_s,  nan=0.0, posinf=1.0, neginf=0.0)
print('Scaled range:', round(float(X_train_s.min()),3), '->', round(float(X_train_s.max()),3))

In [ ]:
# Step 9 - SMOTE to balance the training set
if SMOTE is None:
    raise ImportError('imbalanced-learn is missing. Run the pip install cell in Section 1.')
print('Before SMOTE:', dict(zip(*np.unique(y_train, return_counts=True))))
X_train_bal, y_train_bal = SMOTE(random_state=RANDOM_STATE).fit_resample(X_train_s, y_train)
X_train_bal = X_train_bal.astype(np.float32)
print('After  SMOTE:', dict(zip(*np.unique(y_train_bal, return_counts=True))))
N_FEATURES = X_train_bal.shape[1]
print('N_FEATURES =', N_FEATURES)

---
## 4. Exploratory Data Analysis

In [ ]:
# 4.1 Class distribution (count + percentage)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
vc = df['Label'].value_counts()
palette = sns.color_palette('viridis', len(vc))
ax[0].bar(vc.index, vc.values, color=palette)
ax[0].set_title('Class distribution (count)', fontweight='bold')
ax[0].set_ylabel('Samples')
for i, v in enumerate(vc.values):
    ax[0].text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
ax[1].pie(vc.values, labels=vc.index, autopct='%1.1f%%', colors=palette, startangle=90,
          wedgeprops=dict(width=0.45))
ax[1].set_title('Class proportion', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 4.2 Distributions of a few key features by class
key = [c for c in ['Flow Duration','Tot Fwd Pkts','Flow Pkts/s','Fwd Pkt Len Mean',
                   'Pkt Len Mean','Down/Up Ratio'] if c in feature_cols][:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), key):
    for cls in CLASS_NAMES:
        vals = df.loc[df['Label'] == cls, col]
        vals = vals[(vals > vals.quantile(0.01)) & (vals < vals.quantile(0.99))]
        sns.kdeplot(vals, ax=ax, label=cls, fill=True, alpha=0.25, warn_singular=False)
    ax.set_title(col, fontsize=10); ax.legend(fontsize=7)
plt.suptitle('Feature distributions by class (1-99 percentile)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 4.3 Correlation heatmap (top variance features)
var_top = df[feature_cols].var().sort_values(ascending=False).head(25).index.tolist()
plt.figure(figsize=(13, 10))
sns.heatmap(df[var_top].corr(), cmap='coolwarm', center=0, square=True,
            linewidths=.3, cbar_kws={'shrink': .7})
plt.title('Correlation heatmap (top-25 variance features)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 4.4 PCA 2D projection (sampled)
idx = np.random.choice(len(X_train_s), size=min(8000, len(X_train_s)), replace=False)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
proj = pca.fit_transform(X_train_s[idx])
plt.figure(figsize=(9, 7))
for c in range(NUM_CLASSES):
    m = y_train[idx] == c
    plt.scatter(proj[m, 0], proj[m, 1], s=6, alpha=0.4, label=CLASS_NAMES[c])
plt.legend(); plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title(f'PCA 2D projection (explained var = {pca.explained_variance_ratio_.sum():.1%})',
          fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5. Traditional Machine Learning Models

> **Run order:** complete Sections 1–3 first. This section defines `register`, `fit_eval`, and trains all 12 traditional models.

All metrics are collected in a shared `RESULTS` registry so every model (traditional, deep,
custom, federated, RL) can be compared on the same axes later.

In [ ]:
# ---- prerequisite check (clear message if earlier cells were skipped) ----
def _require(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            'Missing: ' + ', '.join(missing) +
            '. Run Sections 1–3 (setup + preprocessing) first, or use Kernel -> Restart & Run All.')

_require('X_train_bal', 'y_train_bal', 'X_test_s', 'y_test', 'TM_DIR', 'RANDOM_STATE')

# ---- results registry + ML helpers ----
RESULTS = {}

def register(name, y_true, y_pred, proba=None, train_time=None, kind='Traditional'):
    acc = accuracy_score(y_true, y_pred)
    RESULTS[name] = {
        'Accuracy':  acc,
        'Precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall':    recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1':        f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'F1_macro':  f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Train_s':   train_time,
        'kind':      kind,
        'y_pred':    np.asarray(y_pred),
        'proba':     None if proba is None else np.asarray(proba),
    }
    r = RESULTS[name]
    print(f"{name:<22} acc={r['Accuracy']:.4f}  f1={r['F1']:.4f}  "
          f"f1_macro={r['F1_macro']:.4f}  time={train_time if train_time else 0:.1f}s")
    return acc

from sklearn.model_selection import train_test_split as _tts
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              GradientBoostingClassifier, AdaBoostClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

def fit_eval(name, model, Xtr, ytr, full=True, proba=True):
    t = time.time()
    model.fit(Xtr, ytr)
    dt = time.time() - t
    yp = model.predict(X_test_s)
    pr = model.predict_proba(X_test_s) if (proba and hasattr(model, 'predict_proba')) else None
    register(name, y_test, yp, pr, dt, kind='Traditional')
    joblib.dump(model, os.path.join(TM_DIR, name.replace(' ', '_').lower() + '.joblib'))
    return model

# Slow estimators train on a capped subsample; tree/boosting use the full balanced set
SUB = 60000
if len(X_train_bal) > SUB:
    Xs, _, ys, _ = _tts(X_train_bal, y_train_bal, train_size=SUB,
                        stratify=y_train_bal, random_state=RANDOM_STATE)
else:
    Xs, ys = X_train_bal, y_train_bal

print('ML framework ready.')
print('Full balanced set:', X_train_bal.shape, '| subsample for slow models:', Xs.shape)

In [ ]:
# --- 12 traditional models ---
logreg = fit_eval('Logistic Regression',
                  LogisticRegression(max_iter=300, n_jobs=-1), Xs, ys)
nb     = fit_eval('Gaussian NB', GaussianNB(), X_train_bal, y_train_bal)
knn    = fit_eval('KNN', KNeighborsClassifier(n_neighbors=7, n_jobs=-1), Xs, ys)
dtree  = fit_eval('Decision Tree',
                  DecisionTreeClassifier(max_depth=20, random_state=RANDOM_STATE),
                  X_train_bal, y_train_bal)
svm    = fit_eval('Linear SVM',
                  CalibratedClassifierCV(LinearSVC(max_iter=2000), cv=3), Xs, ys)

In [ ]:
rf = fit_eval('Random Forest',
              RandomForestClassifier(n_estimators=200, max_depth=24,
                                     n_jobs=-1, random_state=RANDOM_STATE),
              X_train_bal, y_train_bal)
extra = fit_eval('Extra Trees',
                 ExtraTreesClassifier(n_estimators=200, n_jobs=-1,
                                      random_state=RANDOM_STATE),
                 X_train_bal, y_train_bal)
ada = fit_eval('AdaBoost',
               AdaBoostClassifier(n_estimators=150, random_state=RANDOM_STATE),
               Xs, ys)
gb = fit_eval('Gradient Boosting',
              GradientBoostingClassifier(n_estimators=100, max_depth=5,
                                         random_state=RANDOM_STATE),
              Xs, ys)

In [ ]:
# Gradient boosting libraries (GPU-accelerated where supported)
_require('fit_eval', 'torch', 'xgb', 'lgb', 'CatBoostClassifier')

xgb_clf = xgb.XGBClassifier(
    n_estimators=400, max_depth=8, learning_rate=0.1,
    tree_method='hist', device='cuda' if GPU_OK else 'cpu',
    eval_metric='mlogloss', random_state=RANDOM_STATE)
fit_eval('XGBoost', xgb_clf, X_train_bal, y_train_bal)

try:
    lgb_clf = lgb.LGBMClassifier(n_estimators=400, max_depth=-1, learning_rate=0.1,
                                 device='gpu' if GPU_OK else 'cpu',
                                 random_state=RANDOM_STATE, verbose=-1)
    fit_eval('LightGBM', lgb_clf, X_train_bal, y_train_bal)
except Exception as e:
    print('LightGBM GPU unavailable, falling back to CPU:', str(e)[:80])
    lgb_clf = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.1,
                                 random_state=RANDOM_STATE, verbose=-1)
    fit_eval('LightGBM', lgb_clf, X_train_bal, y_train_bal)

cat_clf = CatBoostClassifier(iterations=400, depth=8, learning_rate=0.1,
                             task_type='GPU' if GPU_OK else 'CPU',
                             verbose=0, random_seed=RANDOM_STATE)
fit_eval('CatBoost', cat_clf, X_train_bal, y_train_bal)

In [ ]:
# Feature importance from the Random Forest
imp = pd.DataFrame({'feature': feature_cols, 'importance': rf.feature_importances_})
imp = imp.sort_values('importance', ascending=False).head(20)
plt.figure(figsize=(9, 7))
sns.barplot(data=imp, y='feature', x='importance', palette='mako')
plt.title('Top-20 feature importances (Random Forest)', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6. Deep Learning Models (PyTorch, GPU-accelerated)

Eight architectures: **DNN, ResNet-MLP, 1D-CNN, LSTM, GRU, BiLSTM, CNN-LSTM, Transformer**.
Training uses mixed precision (AMP) on the GPU for speed and lower VRAM use.

In [ ]:
# ---- shared DL training / evaluation utilities ----
_require('X_train_bal', 'y_train_bal', 'X_test_s', 'y_test', 'N_FEATURES', 'NUM_CLASSES', 'register')

DL_EPOCHS = 12
DL_BATCH  = 512

# tensors stay on CPU; DataLoader streams mini-batches to the GPU
Xtr_t = torch.tensor(X_train_bal)
ytr_t = torch.tensor(y_train_bal)
Xte_t = torch.tensor(X_test_s)
yte_t = torch.tensor(y_test)

n_val = int(0.1 * len(Xtr_t))
perm  = torch.randperm(len(Xtr_t))
val_idx, tr_idx = perm[:n_val], perm[n_val:]
train_ds = TensorDataset(Xtr_t[tr_idx], ytr_t[tr_idx])
val_ds   = TensorDataset(Xtr_t[val_idx], ytr_t[val_idx])
test_ds  = TensorDataset(Xte_t, yte_t)

train_loader = DataLoader(train_ds, batch_size=DL_BATCH, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1024, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=1024, shuffle=False)

cls_counts = np.bincount(y_train_bal, minlength=NUM_CLASSES)
class_w = torch.tensor((cls_counts.sum() / (NUM_CLASSES * cls_counts)), dtype=torch.float32).to(DEVICE)

DL_HISTORY = {}

def train_torch(model, name, epochs=DL_EPOCHS, lr=1e-3, seq=False):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    crit = nn.CrossEntropyLoss(weight=class_w)
    use_amp = (DEVICE.type == 'cuda')
    scaler_amp = torch.amp.GradScaler('cuda', enabled=use_amp)
    hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_state, best_val = None, 1e9; patience, bad = 4, 0
    t0 = time.time()
    for ep in range(epochs):
        model.train(); tl = 0.0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE, non_blocking=True)
            if seq: xb = xb.unsqueeze(-1)            # (B, F, 1)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=use_amp):
                out = model(xb); loss = crit(out, yb)
            scaler_amp.scale(loss).backward()
            scaler_amp.step(opt); scaler_amp.update()
            tl += loss.item() * len(xb)
        tl /= len(train_ds)
        # validation
        model.eval(); vl = 0.0; correct = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE); yb = yb.to(DEVICE)
                if seq: xb = xb.unsqueeze(-1)
                out = model(xb); vl += crit(out, yb).item() * len(xb)
                correct += (out.argmax(1) == yb).sum().item()
        vl /= len(val_ds); va = correct / len(val_ds)
        sched.step(vl)
        hist['train_loss'].append(tl); hist['val_loss'].append(vl); hist['val_acc'].append(va)
        print(f'  [{name}] epoch {ep+1:02d}/{epochs}  train_loss={tl:.4f}  val_loss={vl:.4f}  val_acc={va:.4f}')
        if vl < best_val - 1e-4:
            best_val, best_state, bad = vl, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience:
                print('  early stop'); break
    if best_state: model.load_state_dict(best_state)
    DL_HISTORY[name] = hist
    return model, time.time() - t0

@torch.no_grad()
def eval_torch(model, name, train_time, seq=False, kind='Deep Learning'):
    model.eval(); preds = []; probs = []
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        if seq: xb = xb.unsqueeze(-1)
        out = model(xb)
        p = F.softmax(out, dim=1)
        probs.append(p.cpu().numpy()); preds.append(out.argmax(1).cpu().numpy())
    preds = np.concatenate(preds); probs = np.concatenate(probs)
    register(name, y_test, preds, probs, train_time, kind=kind)
    torch.save({'state_dict': model.state_dict(), 'class_names': CLASS_NAMES,
                'n_features': N_FEATURES, 'arch': name},
               os.path.join(DL_DIR, name.replace(' ', '_').lower() + '.pt'))
    return preds

In [ ]:
# ---------- model definitions ----------
class DNN(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_out))
    def forward(self, x): return self.net(x)

class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(d, d), nn.BatchNorm1d(d), nn.ReLU(),
                               nn.Linear(d, d), nn.BatchNorm1d(d))
    def forward(self, x): return F.relu(x + self.f(x))

class ResMLP(nn.Module):
    def __init__(self, n_in, n_out, d=128):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(n_in, d), nn.BatchNorm1d(d), nn.ReLU())
        self.blocks = nn.Sequential(ResBlock(d), ResBlock(d), ResBlock(d))
        self.out = nn.Linear(d, n_out)
    def forward(self, x): return self.out(self.blocks(self.inp(x)))

class CNN1D(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.c = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128, 64), nn.ReLU(),
                                nn.Dropout(0.3), nn.Linear(64, n_out))
    def forward(self, x):                       # x: (B, F, 1) -> (B, 1, F)
        return self.fc(self.c(x.transpose(1, 2)))

class RNNNet(nn.Module):
    def __init__(self, n_out, cell='lstm', hidden=64, bi=False):
        super().__init__()
        rnn = {'lstm': nn.LSTM, 'gru': nn.GRU}[cell]
        self.rnn = rnn(1, hidden, num_layers=2, batch_first=True,
                       dropout=0.2, bidirectional=bi)
        d = hidden * (2 if bi else 1)
        self.fc = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Dropout(0.3),
                                nn.Linear(64, n_out))
    def forward(self, x):                       # x: (B, F, 1)
        o, _ = self.rnn(x)
        return self.fc(o[:, -1])

class CNN_LSTM(nn.Module):
    def __init__(self, n_out):
        super().__init__()
        self.c = nn.Sequential(nn.Conv1d(1, 32, 3, padding=1), nn.ReLU(),
                               nn.Conv1d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool1d(2))
        self.lstm = nn.LSTM(64, 64, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
                                nn.Linear(64, n_out))
    def forward(self, x):                       # (B, F, 1)
        z = self.c(x.transpose(1, 2)).transpose(1, 2)
        o, _ = self.lstm(z)
        return self.fc(o[:, -1])

class TransformerNet(nn.Module):
    def __init__(self, n_in, n_out, d=64, heads=4, layers=2):
        super().__init__()
        self.proj = nn.Linear(1, d)
        self.pos = nn.Parameter(torch.randn(1, n_in, d) * 0.02)
        enc = nn.TransformerEncoderLayer(d, heads, d * 2, dropout=0.1,
                                         batch_first=True, activation='gelu')
        self.enc = nn.TransformerEncoder(enc, layers)
        self.fc = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_out))
    def forward(self, x):                       # (B, F, 1)
        z = self.proj(x) + self.pos
        z = self.enc(z).mean(1)
        return self.fc(z)

print('DL model classes defined.')

In [ ]:
# ---- train all 8 DL models ----
m, t = train_torch(DNN(N_FEATURES, NUM_CLASSES), 'DNN');              eval_torch(m, 'DNN', t)
m, t = train_torch(ResMLP(N_FEATURES, NUM_CLASSES), 'ResNet-MLP');    eval_torch(m, 'ResNet-MLP', t)
m, t = train_torch(CNN1D(N_FEATURES, NUM_CLASSES), 'CNN1D', seq=True);eval_torch(m, 'CNN1D', t, seq=True)
m, t = train_torch(RNNNet(NUM_CLASSES, 'lstm'), 'LSTM', seq=True);    eval_torch(m, 'LSTM', t, seq=True)
m, t = train_torch(RNNNet(NUM_CLASSES, 'gru'),  'GRU',  seq=True);    eval_torch(m, 'GRU', t, seq=True)
m, t = train_torch(RNNNet(NUM_CLASSES,'lstm',bi=True),'BiLSTM',seq=True); eval_torch(m,'BiLSTM',t,seq=True)
m, t = train_torch(CNN_LSTM(NUM_CLASSES), 'CNN-LSTM', seq=True);      eval_torch(m, 'CNN-LSTM', t, seq=True)
m, t = train_torch(TransformerNet(N_FEATURES, NUM_CLASSES), 'Transformer', seq=True)
eval_torch(m, 'Transformer', t, seq=True)

In [ ]:
# DL training curves
n = len(DL_HISTORY)
fig, axes = plt.subplots(2, n, figsize=(3.2*n, 6))
for j, (name, h) in enumerate(DL_HISTORY.items()):
    axes[0, j].plot(h['train_loss'], label='train'); axes[0, j].plot(h['val_loss'], label='val')
    axes[0, j].set_title(name, fontsize=9); axes[0, j].legend(fontsize=7)
    axes[1, j].plot(h['val_acc'], color='green')
    axes[1, j].set_xlabel('epoch'); axes[1, j].set_ylim(0, 1)
axes[0, 0].set_ylabel('loss'); axes[1, 0].set_ylabel('val acc')
plt.suptitle('Deep learning training curves', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 8. Model Comparison Dashboard

Every model registered in `RESULTS` is compared on the same metrics, plus confusion matrices,
ROC curves, a radar chart, and a per-class F1 heatmap.

In [ ]:
if not RESULTS:
    raise RuntimeError('No models in RESULTS yet. Train models in Sections 5–7 first.')

def _build_res_df():
    rows = []
    for name, r in RESULTS.items():
        rows.append({'Model': name, 'Kind': r['kind'], 'Accuracy': r['Accuracy'],
                     'Precision': r['Precision'], 'Recall': r['Recall'],
                     'F1': r['F1'], 'F1_macro': r['F1_macro'],
                     'Train_s': r['Train_s'] if r['Train_s'] else 0})
    return pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)

res_df = _build_res_df()
try:
    display(res_df.style.background_gradient(
        cmap='Greens', subset=['Accuracy','Precision','Recall','F1','F1_macro']
    ).format({'Accuracy':'{:.4f}','Precision':'{:.4f}','Recall':'{:.4f}',
              'F1':'{:.4f}','F1_macro':'{:.4f}','Train_s':'{:.1f}'}))
except Exception:
    display(res_df)

In [ ]:
# 8.1 Grouped bar: accuracy & F1 by model, colored by paradigm
kind_color = {'Traditional': '#3498db', 'Deep Learning': '#e67e22',
              'Custom': '#2ecc71', 'Federated': '#9b59b6', 'RL': '#e74c3c'}
d = res_df.sort_values('F1')
fig, ax = plt.subplots(figsize=(11, max(6, 0.4*len(d))))
yp = np.arange(len(d))
ax.barh(yp, d['F1'], color=[kind_color.get(k, '#888') for k in d['Kind']])
ax.set_yticks(yp); ax.set_yticklabels(d['Model'])
for i, (f, a) in enumerate(zip(d['F1'], d['Accuracy'])):
    ax.text(f + .002, i, f'F1={f:.3f} | acc={a:.3f}', va='center', fontsize=8)
ax.set_xlim(min(0.5, d['F1'].min()-0.05), 1.02)
ax.set_xlabel('Weighted F1'); ax.set_title('Model performance (color = paradigm)', fontweight='bold')
handles = [plt.Rectangle((0,0),1,1,color=c) for c in kind_color.values()]
ax.legend(handles, kind_color.keys(), loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 8.2 Accuracy vs training time trade-off
plt.figure(figsize=(10, 6))
for k in res_df['Kind'].unique():
    s = res_df[res_df['Kind'] == k]
    plt.scatter(s['Train_s'], s['Accuracy'], s=90, label=k,
                color=kind_color.get(k, '#888'), edgecolor='k', alpha=0.8)
    for _, row in s.iterrows():
        plt.annotate(row['Model'], (row['Train_s'], row['Accuracy']),
                     fontsize=7, xytext=(4, 4), textcoords='offset points')
plt.xscale('symlog'); plt.xlabel('Training time (s, symlog)'); plt.ylabel('Accuracy')
plt.title('Accuracy vs training cost', fontweight='bold'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 8.3 Confusion matrices for the top-4 models
top4 = res_df['Model'].head(4).tolist()
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, name in zip(axes, top4):
    cm = confusion_matrix(y_test, RESULTS[name]['y_pred'])
    cmn = cm.astype(float) / cm.sum(1, keepdims=True)
    sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'{name}\n(acc={RESULTS[name]["Accuracy"]:.3f})', fontsize=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.suptitle('Normalized confusion matrices (top-4 models)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# 8.4 ROC curves (one-vs-rest) for the best model that has probabilities
best = next((n for n in res_df['Model'] if RESULTS[n]['proba'] is not None), None)
if best:
    from sklearn.preprocessing import label_binarize
    proba = RESULTS[best]['proba']
    yb = label_binarize(y_test, classes=list(range(NUM_CLASSES)))
    plt.figure(figsize=(8, 7))
    for c in range(NUM_CLASSES):
        fpr, tpr, _ = roc_curve(yb[:, c], proba[:, c])
        plt.plot(fpr, tpr, label=f'{CLASS_NAMES[c]} (AUC={auc(fpr,tpr):.3f})')
    plt.plot([0,1],[0,1],'k--', alpha=.4)
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'ROC curves — {best}', fontweight='bold'); plt.legend()
    plt.tight_layout(); plt.show()

In [ ]:
# 8.5 Radar chart comparing the top-5 models across metrics
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'F1_macro']
top5 = res_df.head(5)
angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]
fig = plt.figure(figsize=(8, 8)); ax = plt.subplot(111, polar=True)
for _, row in top5.iterrows():
    vals = [row[m] for m in metrics]; vals += vals[:1]
    ax.plot(angles, vals, label=row['Model']); ax.fill(angles, vals, alpha=0.05)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics)
ax.set_ylim(min(0.8, top5[metrics].min().min()-0.02), 1.0)
ax.set_title('Top-5 models — metric radar', fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# 8.6 Per-class F1 heatmap across all models
per_class = {}
for name, r in RESULTS.items():
    f1c = f1_score(y_test, r['y_pred'], average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    per_class[name] = f1c
pc_df = pd.DataFrame(per_class, index=CLASS_NAMES).T.loc[res_df['Model']]
plt.figure(figsize=(8, max(6, 0.4*len(pc_df))))
sns.heatmap(pc_df, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.5, vmax=1.0,
            cbar_kws={'label': 'F1'})
plt.title('Per-class F1 by model', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 9. Federated Learning — Real Multi-Process FedAvg (Flower)

**Not a simulation.** 3 client OS processes + 1 server. Private non-IID shards; only weights leave clients.

**Model:** ResNet-MLP (best DL for FedAvg). XGBoost wins overall F1 but cannot FedAvg.
Live proof: `federated/status.json`, client status files, and the web UI live panel.



In [ ]:
# 9.1 Partition data into private, non-IID client shards
FL_NUM_CLIENTS = 3
os.makedirs(os.path.join(FL_DIR, 'data'), exist_ok=True)

# Non-IID: give each client a skewed class mixture (Dirichlet partition)
rng = np.random.default_rng(RANDOM_STATE)
alpha = 0.5
idx_by_class = [np.where(y_train_bal == c)[0] for c in range(NUM_CLASSES)]
client_idx = [[] for _ in range(FL_NUM_CLIENTS)]
for c in range(NUM_CLASSES):
    rng.shuffle(idx_by_class[c])
    props = rng.dirichlet([alpha] * FL_NUM_CLIENTS)
    splits = (np.cumsum(props) * len(idx_by_class[c])).astype(int)[:-1]
    for i, part in enumerate(np.split(idx_by_class[c], splits)):
        client_idx[i].extend(part.tolist())

for i in range(FL_NUM_CLIENTS):
    ix = np.array(client_idx[i]); rng.shuffle(ix)
    np.savez_compressed(os.path.join(FL_DIR, 'data', f'client_{i}.npz'),
                        X=X_train_bal[ix], y=y_train_bal[ix])
    dist = np.bincount(y_train_bal[ix], minlength=NUM_CLASSES)
    print(f'client_{i}: {len(ix):>7,} samples | class mix {dist.tolist()}')

np.savez_compressed(os.path.join(FL_DIR, 'data', 'test.npz'), X=X_test_s, y=y_test)
json.dump({'n_features': int(N_FEATURES), 'num_classes': int(NUM_CLASSES),
           'class_names': CLASS_NAMES},
          open(os.path.join(FL_DIR, 'data', 'meta.json'), 'w'))
print('Saved shards + test set + meta.json to', os.path.join(FL_DIR, 'data'))

In [ ]:
# 9.2 Refresh FL launcher for exactly 3 clients (server/client/fl_model already shipped)
lines = ['@echo off', 'cd /d "%~dp0"', 'set FL_ROUNDS=5', 'set FL_CLIENTS=3',
         'start "FL-SERVER" cmd /k python server.py', 'timeout /t 5 >nul']
for i in range(FL_NUM_CLIENTS):
    lines.append('start "FL-CLIENT-%d" cmd /k python client.py %d' % (i, i))
    lines.append('timeout /t 1 >nul')
lines += ['echo All processes launched.', 'pause']
with open(os.path.join(FL_DIR, 'run_federated.bat'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines) + '\n')
print('FL ready in', FL_DIR, '| clients:', FL_NUM_CLIENTS)
print('Launch: federated/run_federated.bat')



In [ ]:
# 9.3 Optional: launch FL from notebook (prefer .bat on Windows)
RUN_FL = False
if RUN_FL:
    import subprocess, sys, time
    env = os.environ.copy(); env['FL_ROUNDS']='5'; env['FL_CLIENTS']=str(FL_NUM_CLIENTS)
    srv = subprocess.Popen([sys.executable, 'server.py'], cwd=FL_DIR, env=env)
    time.sleep(5)
    cli = [subprocess.Popen([sys.executable, 'client.py', str(i)], cwd=FL_DIR, env=env) for i in range(FL_NUM_CLIENTS)]
    srv.wait(); [c.wait() for c in cli]
    print('Federated run finished.')
else:
    print('RUN_FL=False. Use federated/run_federated.bat')



In [ ]:
# 9.4 Score federated global model on held-out test set
gpath = os.path.join(FL_DIR, 'global_model.pt')
alt = os.path.join(FL_SAVE, 'federated_fedavg.pt')
path = gpath if os.path.exists(gpath) else alt
if os.path.exists(path):
    import importlib.util
    spec = importlib.util.spec_from_file_location('fl_model', os.path.join(FL_DIR, 'fl_model.py'))
    fl_model = importlib.util.module_from_spec(spec); spec.loader.exec_module(fl_model)
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    sd = ck.get('state_dict', ck)
    n_in = int(sd['inp.0.weight'].shape[1]) if 'inp.0.weight' in sd else X_test_s.shape[1]
    n_out = int(sd['out.weight'].shape[0]) if 'out.weight' in sd else NUM_CLASSES
    gmodel = fl_model.FLNet(n_in=n_in, n_out=n_out)
    gmodel.load_state_dict(sd); gmodel.to(DEVICE).eval()
    X_eval = X_test_s
    if X_eval.shape[1] != n_in:
        if X_eval.shape[1] > n_in: X_eval = X_eval[:, :n_in]
        else: X_eval = np.hstack([X_eval, np.zeros((len(X_eval), n_in-X_eval.shape[1]), np.float32)])
    with torch.no_grad():
        out = gmodel(torch.tensor(X_eval).to(DEVICE))
        fl_pred = out.argmax(1).cpu().numpy()
        fl_proba = F.softmax(out, 1).cpu().numpy()
    register('Federated (FedAvg)', y_test, fl_pred, fl_proba, None, kind='Federated')
    payload = {'state_dict': sd, 'round': ck.get('round'), 'acc': ck.get('acc'),
               'arch': 'ResNet-MLP', 'n_features': n_in, 'class_names': CLASS_NAMES}
    torch.save(payload, os.path.join(FL_SAVE, 'federated_fedavg.pt'))
    torch.save(payload, os.path.join(DL_DIR, 'federated_fedavg.pt'))
    print('Federated model scored. Saved to', FL_SAVE)
else:
    print('No global_model.pt yet. Run federated/run_federated.bat first.')



---
## 10. Reinforcement Learning — 5 Agents (from COMPLETE_RL_FULL_CODE)

Same algorithms, training loop, state/action/reward, and bandit formulation as `COMPLETE_RL_FULL_CODE.ipynb`,
trained on this notebook's **42 pruned features** (`X_train_s` / `X_test_s`).

| Agent | Idea |
|-------|------|
| DQN-Basic | MLP Q-network |
| DQN-Improved | Dueling (value + advantage) |
| PPO | Actor-Critic with clipped surrogate |
| Rainbow-DQN | Dueling + NoisyNets |
| QR-DQN | 51 return quantiles per action |

**Formulation:** contextual bandit (`gamma = 0`) — each flow is an independent state; target = immediate reward.

**Reward (cost-sensitive):** correct action → inverse-frequency class weight; wrong → `-1`.


In [ ]:
# RL — 5 models from COMPLETE_RL_FULL_CODE.ipynb (42-feature pipeline)
# REINFORCEMENT LEARNING MODELS — CSE499B
#   Trains 5 RL agents as flow classifiers. Uses this notebook's existing
#   X_train_s / X_test_s / y_train / y_test (42 pruned features, same split)
#   so RL matches COMPLETE_RL_FULL_CODE except for the feature count.
# MODELS
#   1. Basic DQN      — value-based baseline (MLP estimating Q-values)
#   2. Dueling DQN    — splits Q into state-value + per-action advantage
#   3. PPO            — policy-based; explores via a stochastic (sampled) policy
#   4. Rainbow DQN    — dueling architecture + noisy-net exploration
#   5. QR-DQN         — distributional; predicts 51 return quantiles per action
# KEY DESIGN CHOICE — cost-sensitive reward
#   Dataset is highly imbalanced (Bot/Prob rare). Each class is weighted
#   inversely to its frequency so a correct rare-class prediction is worth far
#   more than a correct common one. Correct action -> class weight ; wrong -> -1.
#
#   Weight formula (standard inverse-frequency / "balanced" scheme):
#       CLASS_W[c] = N_total / (N_CLASSES * count[c])
#   Formulation is a contextual bandit (gamma = 0), not sequential RL —
#   each flow is independent; target = immediate reward.
print(">>> RL cell (5 models) started — setting up ...", flush=True)

_require('X_train_s', 'X_test_s', 'y_train', 'y_test', 'DEVICE', 'RL_DIR',
         'CLASS_NAMES', 'NUM_CLASSES', 'register', 'RANDOM_STATE')

SEED = RANDOM_STATE if 'RANDOM_STATE' in dir() else 42
np.random.seed(SEED); torch.manual_seed(SEED)
print(f">>> device: {DEVICE}", flush=True)

# data from this notebook's 42-feature pipeline (no SMOTE — same setup as COMPLETE_RL)
Xtr = torch.tensor(np.ascontiguousarray(np.asarray(X_train_s, np.float32)))
Xte = torch.tensor(np.ascontiguousarray(np.asarray(X_test_s,  np.float32)))
ytr = torch.tensor(np.asarray(y_train).astype(np.int64).ravel())
yte_np = np.asarray(y_test).astype(np.int64).ravel()

N_FEATURES_RL = Xtr.shape[1]
N_CLASSES_RL  = int(ytr.max().item()) + 1
print(f">>> features={N_FEATURES_RL}  classes={N_CLASSES_RL}  train={len(Xtr)}  test={len(Xte)}", flush=True)
assert N_FEATURES_RL == 42, f"Expected 42 features, got {N_FEATURES_RL}"

# cost-sensitive reward weights (rare classes worth more when correct)
counts = torch.bincount(ytr, minlength=N_CLASSES_RL).float()
CLASS_W = (len(ytr) / (N_CLASSES_RL * counts.clamp(min=1))).to(DEVICE)
print(f">>> reward class-weights: {dict(zip(CLASS_NAMES, np.round(CLASS_W.cpu().numpy(), 3)))}", flush=True)

Xtr_d, ytr_d = Xtr.to(DEVICE), ytr.to(DEVICE)

def batch_reward(actions, labels):
    correct = (actions == labels)
    return torch.where(correct, CLASS_W[labels], torch.tensor(-1.0, device=DEVICE))

EPOCHS = 2
BATCH  = 2048

# ════════════════════════════════════════════════════════════════════════════
# 1-2. DQN family nets (Basic, Dueling/Improved)
# ════════════════════════════════════════════════════════════════════════════
class MLPNet(nn.Module):
    def __init__(s, d, a):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d,256), nn.ReLU(), nn.Linear(256,128), nn.ReLU(), nn.Linear(128,a))
    def forward(s, x): return s.net(x)

class DuelingNet(nn.Module):
    def __init__(s, d, a):
        super().__init__()
        s.f = nn.Sequential(nn.Linear(d,256), nn.ReLU(), nn.LayerNorm(256), nn.Linear(256,128), nn.ReLU(), nn.LayerNorm(128))
        s.v = nn.Sequential(nn.Linear(128,64), nn.ReLU(), nn.Linear(64,1))
        s.a = nn.Sequential(nn.Linear(128,64), nn.ReLU(), nn.Linear(64,a))
    def forward(s, x):
        f = s.f(x); v = s.v(f); a = s.a(f)
        return v + a - a.mean(1, keepdim=True)

# ════════════════════════════════════════════════════════════════════════════
# 3. PPO net
# ════════════════════════════════════════════════════════════════════════════
class ActorCritic(nn.Module):
    def __init__(s, d, a):
        super().__init__()
        s.sh = nn.Sequential(nn.Linear(d,256), nn.ReLU(), nn.LayerNorm(256), nn.Linear(256,128), nn.ReLU(), nn.LayerNorm(128))
        s.ac = nn.Sequential(nn.Linear(128,64), nn.ReLU(), nn.Linear(64,a))
        s.cr = nn.Sequential(nn.Linear(128,64), nn.ReLU(), nn.Linear(64,1))
    def forward(s, x):
        h = s.sh(x); return s.ac(h), s.cr(h)

# ════════════════════════════════════════════════════════════════════════════
# 4. Rainbow: Dueling + Noisy Nets  (noisy layers replace epsilon-greedy)
# ════════════════════════════════════════════════════════════════════════════
class NoisyLinear(nn.Module):
    def __init__(s, i, o, sigma=0.017):
        super().__init__()
        s.i, s.o = i, o
        s.w_mu = nn.Parameter(torch.empty(o, i)); s.w_sig = nn.Parameter(torch.full((o,i), sigma))
        s.b_mu = nn.Parameter(torch.empty(o));    s.b_sig = nn.Parameter(torch.full((o,), sigma))
        s.register_buffer('w_eps', torch.zeros(o, i)); s.register_buffer('b_eps', torch.zeros(o))
        std = 1.0/math.sqrt(i); s.w_mu.data.uniform_(-std,std); s.b_mu.data.uniform_(-std,std)
        s.reset_noise()
    def reset_noise(s):
        s.w_eps.normal_(); s.b_eps.normal_()
    def forward(s, x):
        if s.training:
            w = s.w_mu + s.w_sig * s.w_eps; b = s.b_mu + s.b_sig * s.b_eps
        else:
            w, b = s.w_mu, s.b_mu
        return F.linear(x, w, b)

class RainbowNet(nn.Module):
    def __init__(s, d, a):
        super().__init__()
        s.f = nn.Sequential(NoisyLinear(d,256), nn.ReLU(), nn.LayerNorm(256), NoisyLinear(256,128), nn.ReLU(), nn.LayerNorm(128))
        s.v = nn.Sequential(NoisyLinear(128,64), nn.ReLU(), NoisyLinear(64,1))
        s.a = nn.Sequential(NoisyLinear(128,64), nn.ReLU(), NoisyLinear(64,a))
    def forward(s, x):
        f = s.f(x); v = s.v(f); a = s.a(f)
        return v + a - a.mean(1, keepdim=True)
    def reset_noise(s):
        for m in s.modules():
            if isinstance(m, NoisyLinear): m.reset_noise()

# ════════════════════════════════════════════════════════════════════════════
# 5. QR-DQN: distributional — predicts N quantiles of the return per action
# ════════════════════════════════════════════════════════════════════════════
N_QUANT = 51
class QRNet(nn.Module):
    def __init__(s, d, a, nq=N_QUANT):
        super().__init__()
        s.a, s.nq = a, nq
        s.body = nn.Sequential(nn.Linear(d,256), nn.ReLU(), nn.LayerNorm(256), nn.Linear(256,128), nn.ReLU(), nn.LayerNorm(128))
        s.head = nn.Linear(128, a*nq)
    def forward(s, x):
        return s.head(s.body(x)).view(-1, s.a, s.nq)   # (B, actions, quantiles)
    def q(s, x):
        return s.forward(x).mean(2)                      # mean over quantiles -> Q(s,a)

# ════════════════════════════════════════════════════════════════════════════
# TRAINING  (gamma = 0 bandit: target = immediate reward; vectorised minibatches)
# ════════════════════════════════════════════════════════════════════════════
def train_qnet(net, name, lr, eps_decay, noisy=False):
    net = net.to(DEVICE); opt = torch.optim.Adam(net.parameters(), lr=lr)
    eps, eps_min = 1.0, 0.20
    n = len(Xtr_d); t0 = time.time(); step = 0
    print(f"\n{name} training ...", flush=True)
    for ep in range(EPOCHS):
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            idx = perm[i:i+BATCH]; S, lbl = Xtr_d[idx], ytr_d[idx]
            if noisy: net.reset_noise()
            with torch.no_grad():
                greedy = net(S).argmax(1)
                if noisy:                              # noisy nets explore internally
                    actions = greedy
                else:
                    rand = torch.randint(0, N_CLASSES_RL, (len(idx),), device=DEVICE)
                    explore = torch.rand(len(idx), device=DEVICE) < eps
                    actions = torch.where(explore, rand, greedy)
                rewards = batch_reward(actions, lbl)
            qsa = net(S).gather(1, actions.unsqueeze(1)).squeeze(1)
            loss = F.mse_loss(qsa, rewards)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
            step += 1
            if step % 50 == 0:
                acc = (greedy == lbl).float().mean().item()
                print(f"   {name} ep{ep+1} step{step} greedy_acc={acc:.3f} ({time.time()-t0:.0f}s)", flush=True)
            eps = max(eps_min, eps*eps_decay)
    print(f"   {name} done in {time.time()-t0:.0f}s", flush=True)
    return net

def train_ppo(name, lr):
    net = ActorCritic(N_FEATURES_RL, N_CLASSES_RL).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=lr); clip, vf, ent = 0.2, 0.5, 0.01
    n = len(Xtr_d); t0 = time.time(); step = 0
    print(f"\n{name} training ...", flush=True)
    for ep in range(EPOCHS):
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            idx = perm[i:i+BATCH]; S, lbl = Xtr_d[idx], ytr_d[idx]
            with torch.no_grad():
                logits, val = net(S)
                dist = torch.distributions.Categorical(logits=logits)
                a = dist.sample(); old_logp = dist.log_prob(a)
                r = batch_reward(a, lbl)
                adv = r - val.squeeze(1); adv = (adv-adv.mean())/(adv.std()+1e-8)
            for _ in range(4):
                logits, val = net(S)
                dist = torch.distributions.Categorical(logits=logits)
                ratio = torch.exp(dist.log_prob(a) - old_logp)
                s1 = ratio*adv; s2 = torch.clamp(ratio, 1-clip, 1+clip)*adv
                loss = -torch.min(s1,s2).mean() + vf*F.mse_loss(val.squeeze(1), r) - ent*dist.entropy().mean()
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
            step += 1
            if step % 50 == 0:
                acc = (logits.argmax(1) == lbl).float().mean().item()
                print(f"   {name} ep{ep+1} step{step} acc={acc:.3f} ({time.time()-t0:.0f}s)", flush=True)
    print(f"   {name} done in {time.time()-t0:.0f}s", flush=True)
    return net

def train_qrdqn(name, lr, eps_decay):
    net = QRNet(N_FEATURES_RL, N_CLASSES_RL).to(DEVICE); opt = torch.optim.Adam(net.parameters(), lr=lr)
    eps, eps_min = 1.0, 0.05
    # quantile midpoints tau
    tau = ((torch.arange(N_QUANT, device=DEVICE).float() + 0.5) / N_QUANT).view(1, -1)
    n = len(Xtr_d); t0 = time.time(); step = 0
    print(f"\n{name} training ...", flush=True)
    for ep in range(EPOCHS):
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            idx = perm[i:i+BATCH]; S, lbl = Xtr_d[idx], ytr_d[idx]
            with torch.no_grad():
                greedy = net.q(S).argmax(1)
                rand = torch.randint(0, N_CLASSES_RL, (len(idx),), device=DEVICE)
                explore = torch.rand(len(idx), device=DEVICE) < eps
                actions = torch.where(explore, rand, greedy)
                rewards = batch_reward(actions, lbl)                 # (B,)
            # predicted quantiles for the taken action: (B, nq)
            allq = net(S)                                            # (B, A, nq)
            theta = allq.gather(1, actions.view(-1,1,1).expand(-1,1,N_QUANT)).squeeze(1)
            # gamma=0 bandit -> target quantiles all equal the immediate reward
            target = rewards.unsqueeze(1).expand(-1, N_QUANT)        # (B, nq)
            # quantile Huber loss
            u = target - theta                                       # (B, nq)
            huber = 0.5*u.pow(2)
            loss = (torch.abs(tau - (u.detach() < 0).float()) * huber).mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
            step += 1
            if step % 50 == 0:
                acc = (greedy == lbl).float().mean().item()
                print(f"   {name} ep{ep+1} step{step} greedy_acc={acc:.3f} ({time.time()-t0:.0f}s)", flush=True)
            eps = max(eps_min, eps*eps_decay)
    print(f"   {name} done in {time.time()-t0:.0f}s", flush=True)
    return net

def predict(net, kind='dqn'):
    net.eval(); out = np.empty(len(Xte), np.int64); bs = 16384
    with torch.no_grad():
        for i in range(0, len(Xte), bs):
            chunk = Xte[i:i+bs].to(DEVICE)
            if kind == 'ppo':   q = net(chunk)[0]
            elif kind == 'qr':  q = net.q(chunk)
            else:               q = net(chunk)
            out[i:i+bs] = q.argmax(1).cpu().numpy()
    return out

# ── run all 5 ───────────────────────────────────────────────────────────────
t_rl0 = time.time()
basic_net    = train_qnet(MLPNet(N_FEATURES_RL, N_CLASSES_RL),     "DQN-Basic",    1e-3, 0.999)
improved_net = train_qnet(DuelingNet(N_FEATURES_RL, N_CLASSES_RL), "DQN-Improved", 3e-4, 0.999)
ppo_net      = train_ppo("PPO", 3e-4)
rainbow_net  = train_qnet(RainbowNet(N_FEATURES_RL, N_CLASSES_RL), "Rainbow-DQN",  3e-4, 0.999, noisy=True)
qrdqn_net    = train_qrdqn("QR-DQN", 3e-4, 0.999)
rl_time_total = time.time() - t_rl0

dqn_basic_preds    = predict(basic_net,    'dqn')
dqn_improved_preds = predict(improved_net, 'dqn')
ppo_preds          = predict(ppo_net,      'ppo')
rainbow_preds      = predict(rainbow_net,  'dqn')
qrdqn_preds        = predict(qrdqn_net,    'qr')

print("\n" + "="*66 + "\nRL RESULTS (test set)\n" + "="*66, flush=True)
from sklearn.metrics import accuracy_score, f1_score, classification_report
_rl_bundle = [
    ("DQN-Basic",    "dqn_basic",    basic_net,    dqn_basic_preds,    'dqn'),
    ("DQN-Improved", "dqn_improved", improved_net, dqn_improved_preds, 'dqn'),
    ("PPO",          "ppo",          ppo_net,      ppo_preds,          'ppo'),
    ("Rainbow-DQN",  "rainbow_dqn",  rainbow_net,  rainbow_preds,      'dqn'),
    ("QR-DQN",       "qr_dqn",       qrdqn_net,    qrdqn_preds,        'qr'),
]
for nm, stem, net, pr, kind in _rl_bundle:
    acc = accuracy_score(yte_np, pr)
    f1w = f1_score(yte_np, pr, average='weighted', zero_division=0)
    f1m = f1_score(yte_np, pr, average='macro', zero_division=0)
    print(f"  {nm:<14} acc={acc:.4f}  f1_weighted={f1w:.4f}  f1_macro={f1m:.4f}", flush=True)
    register(nm, y_test, pr, None, rl_time_total / 5, kind='RL')
    torch.save({'state_dict': net.state_dict(), 'class_names': CLASS_NAMES,
                'n_features': N_FEATURES_RL, 'arch': nm, 'kind': kind},
               os.path.join(RL_DIR, stem + '.pt'))
print('RL checkpoints saved to', RL_DIR)


In [ ]:
# RL comparison table (same 5 agents as COMPLETE_RL_FULL_CODE)
_rl_names = ['DQN-Basic','DQN-Improved','PPO','Rainbow-DQN','QR-DQN']
_rows = [(n, RESULTS[n]) for n in _rl_names if n in RESULTS]
if _rows:
    display(pd.DataFrame([{'Model': n, 'Accuracy': r['Accuracy'], 'F1': r['F1'], 'F1_macro': r['F1_macro']} for n, r in _rows]))
else:
    print('Run the RL training cell first.')


In [ ]:
# Model selection — pick best RL agent by weighted F1 (same metric COMPLETE reports)
from sklearn.metrics import classification_report
_rl_names = ['DQN-Basic','DQN-Improved','PPO','Rainbow-DQN','QR-DQN']
_present = [(n, RESULTS[n]['F1']) for n in _rl_names if n in RESULTS]
if _present:
    best_rl = max(_present, key=lambda t: t[1])[0]
    pred_map = {'DQN-Basic': dqn_basic_preds, 'DQN-Improved': dqn_improved_preds,
                'PPO': ppo_preds, 'Rainbow-DQN': rainbow_preds, 'QR-DQN': qrdqn_preds}
    print('Best RL agent by weighted F1:', best_rl)
    print(classification_report(y_test, pred_map[best_rl], target_names=CLASS_NAMES, digits=3))
else:
    print('Run the RL training cell first.')


In [ ]:
# RL section complete. Checkpoints in Saved model/RL/
print('Section 10 done — RL agents registered and saved.')


---
## 10B. Federated Learning over RL (RL -> FL)

**Order matters:** finish **Section 10** (local RL) first, then run this.

\Client private flows -> local RL train -> upload RL weights only
Flower FedAvg -> global RL model  (no raw data shared)
\
**Launcher (recommended on Windows):**
\\at
federated\\run_rl_federated.bat
\
Saves: \Saved model/FL/federated_rl_fedavg.ptThen restart \ids_server.py\ and check Predict -> **Federated RL (RL->FL)**.


In [ ]:
# ===== Section 10B — launch RL -> FL (Flower FedAvg of RL weights) =====
# Prerequisite: Section 10 done (Saved model/RL/dqn_improved.pt recommended).
import os
print('RL -> FL')
print('1) Confirm local RL exists:', os.path.exists(os.path.join('Saved model', 'RL', 'dqn_improved.pt')))
print('2) Double-click or run: federated/run_rl_federated.bat')
print('3) Watch the FL page in the UI while rounds run.')
print('4) Restart ids_server.py after FL finishes.')
print('Global RL checkpoint will be: Saved model/FL/federated_rl_fedavg.pt')
print('Section 10B instructions printed — run the bat outside the notebook.')


---
## 11. Save Shared Artifacts
Artifacts go under `Saved model/`.



In [ ]:
joblib.dump(scaler,        os.path.join(ART_DIR, 'scaler.joblib'))
joblib.dump(label_encoder, os.path.join(ART_DIR, 'label_encoder.joblib'))
joblib.dump(feature_cols,  os.path.join(ART_DIR, 'feature_cols.joblib'))
if '_build_res_df' in globals() and RESULTS:
    res_df = _build_res_df()
elif os.path.exists(os.path.join(ART_DIR, 'results_summary.csv')):
    res_df = pd.read_csv(os.path.join(ART_DIR, 'results_summary.csv'))
else:
    res_df = pd.DataFrame()
best_name = res_df.iloc[0]['Model'] if len(res_df) > 0 else 'XGBoost'
manifest = {
    'n_features': int(N_FEATURES), 'num_classes': int(NUM_CLASSES),
    'class_names': CLASS_NAMES, 'feature_columns': feature_cols,
    'best_model': best_name, 'ui_predictor': 'XGBoost',
    'results': {k: {m: float(v[m]) for m in ['Accuracy','Precision','Recall','F1','F1_macro']} for k, v in RESULTS.items()},
}
json.dump(manifest, open(os.path.join(ART_DIR, 'model_info.json'), 'w'), indent=2)
res_df.to_csv(os.path.join(ART_DIR, 'results_summary.csv'), index=False)
print('Saved to', ART_DIR, '| Best:', best_name, '| UI predictor: XGBoost')



---
## 12. Flask Web UI
Run `python ids_server.py` → http://127.0.0.1:7860
Only XGBoost predicts; live FL/RL panel proves real systems.



In [ ]:
print('Start dashboard:  python ids_server.py')
print('Open: http://127.0.0.1:7860')
print('FL: federated/run_federated.bat')



In [ ]:
print('Web stack: ids_server.py + ids_models.py + templates/dashboard.html')



---
## 13. Summary
Supervised best (UI): XGBoost · FL: Flower 1+3 ResNet-MLP · RL: 5 agents · Proof: /live + dual inference
See docs/IDS_Universal_Guide.docx



In [ ]:
print('='*70)
print('Run 1-8 train | 9 FL | 10 RL | 11 save | then python ids_server.py')
print('='*70)

